In [ ]:
"""
Script to run TableGAN on multiple datasets.
"""

import os
import importlib
import traceback
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline

# Import your utility functions
from utils import discretize_preprocess

def ensure(path):
    """Create directory if it doesn't exist."""
    os.makedirs(path, exist_ok=True)

def make_pipeline(model_callable, skip_evaluations=True, evaluations=None):
    """
    Create a pipeline with optional evaluation settings.
    
    Args:
        model_callable: Function that returns model instance
        skip_evaluations: If True, skip TSTR evaluations
        evaluations: List of specific evaluations to run
    """
    if skip_evaluations:
        return TrainTestSplitPipeline(
            model=model_callable, 
            evaluations=[], 
            override_evaluations=True
        )
    elif evaluations is not None:
        return TrainTestSplitPipeline(
            model=model_callable, 
            evaluations=evaluations, 
            override_evaluations=True
        )
    else:
        return TrainTestSplitPipeline(model=model_callable)


# ============================================================================
# Configuration
# ============================================================================

# Datasets to process
DATASETS = ['adult', 'car', 'magic', 'nursery', 'shuttle']

# Set to False to run TSTR evaluations (requires xgboost)
SKIP_EVALUATIONS = False

# Create necessary directories
ensure('discretized_data')
ensure('sample_data')
ensure('synthetic')
ensure('Results')

# TableGAN Configuration
TABLEGAN_CONFIG = {
    'epochs': 300,
    'batch_size': 64,
    'noise_dim': 100,
    'generator_filters': (256, 128, 64),
    'discriminator_filters': (64, 128, 256),
    'generator_lr': 2e-4,
    'discriminator_lr': 2e-4,
    'n_critic': 5,
    'lambda_gp': 10.0,
    'use_gradient_penalty': True,
    'dropout': 0.1,
}


# ============================================================================
# Step 1: Preprocess all datasets
# ============================================================================

print("\n" + "=" * 80)
print("STEP 1: PREPROCESSING DATASETS")
print("=" * 80)

for dataset in DATASETS:
    print(f'\nPreprocessing {dataset}...')
    try:
        discretize_preprocess(
            file_path=f'raw_data/{dataset}.csv',
            output_path=f'discretized_data/{dataset}.csv',
            bins=10,
            strategy='uniform'
        )
        print(f'✓ Discretized -> discretized_data/{dataset}.csv')
    except Exception as e:
        print(f'✗ Failed to preprocess {dataset}: {e}')
        traceback.print_exc()


# ============================================================================
# Step 2: Run TableGAN on all datasets
# ============================================================================

print("\n" + "=" * 80)
print("STEP 2: RUNNING TABLEGAN")
print("=" * 80)

for dataset in DATASETS:
    print(f'\n{"-" * 60}')
    print(f'Running TableGAN on {dataset}...')
    print(f'{"-" * 60}')
    
    synth_dir = os.path.join('synthetic', dataset, 'tablegan')
    ensure(synth_dir)
    
    try:
        # Import TableGAN
        module = importlib.import_module('katabatic.models.tablegan.models')
        TableGAN = getattr(module, 'TableGAN')
        
        # Create model factory
        model_factory = lambda: TableGAN(**TABLEGAN_CONFIG)
        pipeline = make_pipeline(model_factory, skip_evaluations=SKIP_EVALUATIONS)
        
        # Run pipeline
        result = pipeline.run(
            input_csv=f'discretized_data/{dataset}.csv',
            output_dir=f'sample_data/{dataset}',
            synthetic_dir=synth_dir,
            real_test_dir=f'sample_data/{dataset}'
        )
        
        print(f'✓ TableGAN finished for {dataset}')
        if result:
            print(f'  Result: {result}')
    except Exception as e:
        print(f'✗ TableGAN failed for {dataset}: {e}')
        traceback.print_exc()


# ============================================================================
# Summary
# ============================================================================

print("\n" + "=" * 80)
print("TABLEGAN COMPLETED")
print("=" * 80)
print("\nSynthetic data has been generated for all datasets.")
print("Check the following directories:")
for dataset in DATASETS:
    print(f"  - synthetic/{dataset}/tablegan/")
print("\nResults stored in: Results/")


STEP 1: PREPROCESSING DATASETS

Preprocessing adult...
Preprocessing: raw_data/adult.csv
Saved preprocessed discrete dataset to: discretized_data/adult.csv
✓ Discretized -> discretized_data/adult.csv

Preprocessing car...
Preprocessing: raw_data/car.csv
Saved preprocessed discrete dataset to: discretized_data/car.csv
✓ Discretized -> discretized_data/car.csv

Preprocessing magic...
Preprocessing: raw_data/magic.csv
Saved preprocessed discrete dataset to: discretized_data/magic.csv
✓ Discretized -> discretized_data/magic.csv

Preprocessing nursery...
Preprocessing: raw_data/nursery.csv
Saved preprocessed discrete dataset to: discretized_data/nursery.csv
✓ Discretized -> discretized_data/nursery.csv

Preprocessing shuttle...
Preprocessing: raw_data/shuttle.csv
Saved preprocessed discrete dataset to: discretized_data/shuttle.csv
✓ Discretized -> discretized_data/shuttle.csv

STEP 2: RUNNING TABLEGAN

------------------------------------------------------------
Running TableGAN on adult..

INFO:katabatic.models.tablegan.models:================================================================================
INFO:katabatic.models.tablegan.models:Training TableGAN Model
INFO:katabatic.models.tablegan.models:================================================================================
INFO:katabatic.models.tablegan.models:Loaded training data: (26048, 14)


Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)


INFO:katabatic.models.tablegan.models:Table dimensions: 4x4 (padded from 15 features)
INFO:katabatic.models.tablegan.models:Generator parameters: 109,568
INFO:katabatic.models.tablegan.models:Discriminator parameters: 75,201
INFO:katabatic.models.tablegan.models:Epoch 1/300: D Loss = -0.852799, G Loss = -0.604852


In [1]:
# Quick test with just 'car' dataset
DATASETS = ['adult', 'car', 'magic', 'nursery', 'shuttle']

# Or run this minimal test:
import os
import importlib
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess

def ensure(path):
    os.makedirs(path, exist_ok=True)

# Preprocess
print("Preprocessing car dataset...")
discretize_preprocess(
    file_path='raw_data/car.csv',
    output_path='discretized_data/car.csv',
    bins=10,
    strategy='uniform'
)

# Run TableGAN
print("\nRunning TableGAN...")
module = importlib.import_module('katabatic.models.tablegan.models')
TableGAN = getattr(module, 'TableGAN')

# Reduced config for faster testing
config = {
    'epochs': 10,  
    'batch_size': 64,
    'noise_dim': 100,
}

ensure('synthetic/car/tablegan')
model_factory = lambda: TableGAN(**config)
pipeline = TrainTestSplitPipeline(model=model_factory, evaluations=[], override_evaluations=True)

result = pipeline.run(
    input_csv='discretized_data/car.csv',
    output_dir='sample_data/car',
    synthetic_dir='synthetic/car/tablegan',
    real_test_dir='sample_data/car'
)

print("\n✓ Done! Check synthetic/car/tablegan/")

Preprocessing car dataset...
Preprocessing: raw_data/car.csv
Saved preprocessed discrete dataset to: discretized_data/car.csv

Running TableGAN...


INFO:katabatic.models.tablegan.models:================================================================================
INFO:katabatic.models.tablegan.models:Training TableGAN Model
INFO:katabatic.models.tablegan.models:================================================================================
INFO:katabatic.models.tablegan.models:Loaded training data: (1382, 6)
INFO:katabatic.models.tablegan.models:Table dimensions: 4x4 (padded from 7 features)
INFO:katabatic.models.tablegan.models:Generator parameters: 109,568
INFO:katabatic.models.tablegan.models:Discriminator parameters: 75,201


Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
2    0.700434
0    0.222142
1    0.039797
3    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
2    0.699422
0    0.222543
1    0.040462
3    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)


INFO:katabatic.models.tablegan.models:Epoch 1/10: D Loss = -1.135917, G Loss = 0.528112
INFO:katabatic.models.tablegan.models:
Generating 1382 synthetic samples...
INFO:katabatic.models.tablegan.models:
Synthetic data saved to: synthetic/car/tablegan
INFO:katabatic.models.tablegan.models:Training complete!
C:\Users\lbrum\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\lbrum\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [16:57:45] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results\car\tablegan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.1272
F1 Score: 0.0999

MLP:
Accuracy: 0.1098
F1 Score: 0.0852

RF:
Accuracy: 0.1069
F1 Score: 0.1270

XGBoost:
Accuracy: 0.1387
F1 Score: 0.1693

✓ Done! Check synthetic/car/tablegan/


In [3]:
# Quick test with just 'car' dataset
DATASETS = ['car']  # Change this line in the script

# Or run this minimal test:
import os
import importlib
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess

def ensure(path):
    os.makedirs(path, exist_ok=True)

# Preprocess
print("Preprocessing car dataset...")
discretize_preprocess(
    file_path='raw_data/car.csv',
    output_path='discretized_data/car.csv',
    bins=10,
    strategy='uniform'
)

# Run TableGAN
print("\nRunning TableGAN...")
module = importlib.import_module('katabatic.models.tablegan.models')
TableGAN = getattr(module, 'TableGAN')

# Reduced config for faster testing
config = {
    'epochs': 100,  # Reduced from 300
    'batch_size': 64,
    'noise_dim': 100,
}

ensure('synthetic/car/tablegan')
model_factory = lambda: TableGAN(**config)
pipeline = TrainTestSplitPipeline(model=model_factory, evaluations=[], override_evaluations=True)

result = pipeline.run(
    input_csv='discretized_data/car.csv',
    output_dir='sample_data/car',
    synthetic_dir='synthetic/car/tablegan',
    real_test_dir='sample_data/car'
)

print("\n✓ Done! Check synthetic/car/tablegan/")

INFO:katabatic.models.tablegan.models:================================================================================
INFO:katabatic.models.tablegan.models:Training TableGAN Model
INFO:katabatic.models.tablegan.models:================================================================================
INFO:katabatic.models.tablegan.models:Loaded training data: (1382, 6)
INFO:katabatic.models.tablegan.models:Table dimensions: 4x4 (padded from 7 features)
INFO:katabatic.models.tablegan.models:Generator parameters: 109,568
INFO:katabatic.models.tablegan.models:Discriminator parameters: 75,201


Preprocessing car dataset...
Preprocessing: raw_data/car.csv
Saved preprocessed discrete dataset to: discretized_data/car.csv

Running TableGAN...
Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
2    0.700434
0    0.222142
1    0.039797
3    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
2    0.699422
0    0.222543
1    0.040462
3    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)


INFO:katabatic.models.tablegan.models:Epoch 1/100: D Loss = -1.135917, G Loss = 0.528112
INFO:katabatic.models.tablegan.models:Epoch 20/100: D Loss = -126.873721, G Loss = -7.179925
INFO:katabatic.models.tablegan.models:Epoch 40/100: D Loss = -402.232922, G Loss = -27.683669
INFO:katabatic.models.tablegan.models:Epoch 60/100: D Loss = -752.970329, G Loss = -65.351185
INFO:katabatic.models.tablegan.models:Epoch 80/100: D Loss = -1156.739166, G Loss = -123.671381
INFO:katabatic.models.tablegan.models:Epoch 100/100: D Loss = -1591.276686, G Loss = -197.420424
INFO:katabatic.models.tablegan.models:
Generating 1382 synthetic samples...
INFO:katabatic.models.tablegan.models:
Synthetic data saved to: synthetic/car/tablegan
INFO:katabatic.models.tablegan.models:Training complete!
C:\Users\lbrum\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  


Results saved to: Results\car\tablegan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.1214
F1 Score: 0.0818

MLP:
Accuracy: 0.1561
F1 Score: 0.1645

RF:
Accuracy: 0.1734
F1 Score: 0.2241

XGBoost:
Accuracy: 0.1532
F1 Score: 0.2006

✓ Done! Check synthetic/car/tablegan/


C:\Users\lbrum\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:06:46] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
